# T3.A3.1 — Implementación de clasificación y rastreo de objetos con RNC

## Objetivo

Aplicar la arquitectura de una red neuronal convolucional y desarrollar programas basados en este tipo de modelos, mediante la implementación de algoritmos de aprendizaje profundo, en un entorno académico y computacional controlado, con el propósito de aplicar soluciones de visión artificial a problemas representativos del área.

---

## Instrucciones para el participante

1. Ejecuta cada celda **en orden** con `Shift + Enter`.
2. Completa las celdas marcadas con 📝 — son parte de tu portafolio de evidencias.
3. **No borres** las celdas de código base; puedes agregar celdas nuevas si lo necesitas.
4. Al finalizar, exporta este notebook como PDF (`Archivo → Imprimir → Guardar como PDF`).
5. Nombre de entrega: `T3_A3_ApellidoNombre.pdf`

> 💡 **Google Colab (recomendado):** Activa la GPU en `Entorno de ejecución → Cambiar tipo de entorno → GPU` para acelerar el entrenamiento.

---

### Contenido del Tema 3 que cubre esta actividad
- **3.1** Visión monocular
- **3.2** Visión estereoscópica
- **3.3** Arquitecturas de redes convolucionales
- **3.4** Clasificación de imágenes con RNC
- **3.5** Rastreo de imágenes (tracking)

### Conexión con los temas previos
```
Tema 1: Adquisición y representación  →  Tema 2: Filtros y convolución  →  Tema 3: RNC completas (esta actividad)
(imagen como matriz de píxeles)           (operaciones matemáticas)          (clasificación + rastreo)
```

---
## PASO 1 — Selección del problema y contexto
**Tiempo estimado: 40 minutos**

Antes de escribir código, define claramente qué vas a clasificar y rastrear.  
Revisa la **Presentación del Tema 3** en el Aula Virtual TecNM, especialmente los subtemas 3.1 a 3.5.

### Conceptos clave del Tema 3

| Concepto | Definición |
|---|---|
| **Visión monocular** | Sistema que opera con una sola cámara; extrae bordes, texturas y categorías de objetos a partir de imágenes 2D |
| **Visión estereoscópica** | Usa dos cámaras para estimar profundidad mediante el cálculo de disparidad (paralaje) |
| **RNC (CNN)** | Red Neuronal Convolucional: modelo de aprendizaje profundo diseñado para procesar datos en forma de imagen |
| **Capa convolucional** | Aplica filtros (kernels) aprendibles para extraer características locales: bordes, texturas, formas |
| **Pooling** | Reduce la dimensionalidad espacial conservando la información más relevante |
| **Transfer learning** | Reutilizar un modelo preentrenado (LeNet, ResNet, MobileNet) adaptando solo las capas finales |
| **Clasificación** | Asignar una etiqueta a toda la imagen (perro, auto, pieza defectuosa, etc.) |
| **Rastreo (tracking)** | Seguir la posición de uno o varios objetos a lo largo de cuadros sucesivos de un video |
| **Accuracy** | Proporción de aciertos del modelo: TP + TN / Total |
| **Matriz de confusión** | Tabla que muestra verdaderos positivos, falsos positivos, falsos negativos y verdaderos negativos |

### Flujo de un proyecto de clasificación con RNC
```
1. Definir problema  →  2. Dataset  →  3. Preprocesamiento  →  4. Entrenar RNC  →  5. Evaluar  →  6. Rastreo
   (esta celda)          (Paso 3)       (resize, normalizar)    (Paso 4)            (métricas)     (Paso 5)
```

### 📝 Mi problema de trabajo _(completa esta celda antes de continuar)_

**Contexto de aplicación elegido:**  
☐ Industrial (piezas correctas vs. defectuosas) &nbsp;&nbsp; ☐ Tráfico (vehículos, peatones) &nbsp;&nbsp; ☐ Educativo &nbsp;&nbsp; ☐ Otro: ___

**¿Qué objetos voy a clasificar?** ___  
**Clases definidas:** Clase 0 = ___ / Clase 1 = ___ / Clase 2 (si aplica) = ___  
**¿Qué objetos voy a rastrear en video o secuencia de imágenes?** ___  

**Descripción del escenario** _(5–8 líneas)_:  
_Describe el tipo de imágenes o video, el propósito de la aplicación y para qué serviría en la práctica._  

**Conexión con Tema 1 y Tema 2:**  
_¿Qué operaciones de preprocesamiento del Tema 1 o filtros del Tema 2 serán relevantes aquí?_ ___

---
## PASO 2 — Preparación del entorno de trabajo
**Tiempo estimado: 1 hora**

Verifica que las bibliotecas necesarias están disponibles.  
Si trabajas en entorno local y faltan bibliotecas, descomenta e instala:

In [ ]:
# Instalación (descomenta si es necesario)
# !pip install numpy matplotlib opencv-python tensorflow torch torchvision

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os, sys, warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Métricas
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Verificación de versiones
import matplotlib as mpl
print('=' * 50)
print('  ✅ ENTORNO VERIFICADO CORRECTAMENTE')
print('=' * 50)
print(f'  Python      : {sys.version.split()[0]}')
print(f'  NumPy       : {np.__version__}')
print(f'  OpenCV      : {cv2.__version__}')
print(f'  Matplotlib  : {mpl.__version__}')
print(f'  TensorFlow  : {tf.__version__}')
print(f'  Keras       : {keras.__version__}')
gpu = tf.config.list_physical_devices('GPU')
print(f'  GPU         : {"✅ Disponible" if gpu else "⬜ No detectada (CPU)"}')
print('=' * 50)

---
## PASO 3 — Selección / construcción del conjunto de datos
**Tiempo estimado: 1 hora**

Selecciona o construye un conjunto de datos sencillo pero representativo (2–3 clases).  
Puedes usar el dataset **CIFAR-10** (integrado en Keras) como punto de partida, o bien un dataset propio.

> ⚠️ **Dataset propio:** Organiza tus imágenes en carpetas:  
> `train/clase_0/`, `train/clase_1/`, `val/clase_0/`, `val/clase_1/`  
> Luego usa `keras.preprocessing.image_dataset_from_directory()` para cargarlas.

Esta celda usa **CIFAR-10** con 3 clases seleccionadas como ejemplo base. Puedes adaptarla.

In [ ]:
# ─── PARÁMETROS — ajusta según tu problema ───────────────────────────────────
CLASES_ELEGIDAS  = [0, 1, 2]     # Índices de CIFAR-10: 0=avión, 1=auto, 2=pájaro
NOMBRES_CLASES   = ['avión', 'automóvil', 'pájaro']
IMG_SIZE         = (32, 32)      # Tamaño de entrada al modelo
USAR_DATASET_PROPIO = False      # Cambia a True si tienes imágenes propias

# ── Carga del dataset ─────────────────────────────────────────────────────────
if not USAR_DATASET_PROPIO:
    print('Descargando CIFAR-10...')
    (x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.cifar10.load_data()

    # Filtrar solo las clases elegidas
    def filtrar_clases(x, y, clases):
        mascara = np.isin(y.flatten(), clases)
        x_f = x[mascara]
        y_f = y[mascara].flatten()
        # Reasignar etiquetas a 0, 1, 2...
        mapa = {c: i for i, c in enumerate(clases)}
        y_f = np.array([mapa[v] for v in y_f])
        return x_f, y_f

    x_train, y_train = filtrar_clases(x_train_full, y_train_full, CLASES_ELEGIDAS)
    x_test,  y_test  = filtrar_clases(x_test_full,  y_test_full,  CLASES_ELEGIDAS)

    # Normalización: valores de píxel al rango [0.0, 1.0]
    x_train = x_train.astype('float32') / 255.0
    x_test  = x_test.astype('float32')  / 255.0

    print('\nINFORMACIÓN DEL DATASET')
    print(f'  Clases         : {NOMBRES_CLASES}')
    print(f'  Imágenes train : {x_train.shape[0]}')
    print(f'  Imágenes test  : {x_test.shape[0]}')
    print(f'  Dimensiones    : {x_train.shape[1]} × {x_train.shape[2]} px — {x_train.shape[3]} canales')
    print(f'  Tipo de dato   : {x_train.dtype}  → valores en [0.0, 1.0]')
    for i, nombre in enumerate(NOMBRES_CLASES):
        n = (y_train == i).sum()
        print(f'  Clase {i} ({nombre:<12}): {n} imágenes de entrenamiento')

In [ ]:
# ── Visualización de muestras del dataset ─────────────────────────────────────
# Verificar rutas, etiquetas y distribución visual de las clases
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
fig.suptitle('Paso 3 — Muestras del dataset por clase', fontsize=13, fontweight='bold')

for fila, clase in enumerate(range(len(NOMBRES_CLASES))):
    idx = np.where(y_train == clase)[0][:8]
    for col, i in enumerate(idx):
        ax = axes[fila, col]
        ax.imshow(x_train[i])
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(f'Clase {clase}\n{NOMBRES_CLASES[clase]}',
                          fontsize=9, rotation=0, labelpad=55, va='center')

plt.tight_layout()
plt.savefig('paso3_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: paso3_dataset.png')

### 📝 Registro del Paso 3 _(completa esta celda)_

**Nombre / fuente del dataset:** ___  
**Número de clases:** ___  
**Resolución de las imágenes:** ___ × ___ px — ___ canales  
**Imágenes de entrenamiento por clase:** Clase 0 = ___ / Clase 1 = ___ / Clase 2 = ___  
**Imágenes de prueba totales:** ___  
**¿Las muestras visualizadas corresponden a las etiquetas esperadas?** ☐ Sí &nbsp;&nbsp; ☐ No (explica)  
**¿Qué tan balanceado está el dataset entre clases?** ___

---
## PASO 4 — Implementación del modelo de clasificación con RNC
**Tiempo estimado: 2 horas**

Implementa y entrena una CNN para clasificar las imágenes de tu dataset.  
Puedes elegir entre dos opciones:

| Opción | Descripción | Cuándo usarla |
|---|---|---|
| **A — CNN desde cero** | 2–3 capas conv + pooling + capas densas | Dataset propio pequeño o aprendizaje exploratorio |
| **B — Transfer learning** | Modelo preentrenado (MobileNetV2, ResNet…) con capas finales ajustadas | Cuando el dataset es limitado o las clases son complejas |

La celda de abajo implementa la **Opción A**. Al final del paso encontrarás la **Opción B** comentada.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# OPCIÓN A — CNN sencilla construida desde cero
# ═══════════════════════════════════════════════════════════════════════════════

# ─── HIPERPARÁMETROS — modifica y observa los cambios ────────────────────────
EPOCAS      = 15        # Número de pasadas completas por el dataset
BATCH_SIZE  = 64        # Imágenes procesadas simultáneamente
LEARNING_RATE = 0.001   # Tasa de aprendizaje del optimizador Adam
N_CLASES    = len(NOMBRES_CLASES)

# ── Definición de la arquitectura ─────────────────────────────────────────────
# Recuerda (Tema 3, sección 3.3): cada capa aprende filtros adecuados a la tarea
modelo = keras.Sequential([

    # ── Bloque 1: extracción de características de bajo nivel (bordes, texturas)
    layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                  input_shape=(x_train.shape[1], x_train.shape[2], x_train.shape[3])),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),   # Reduce dimensión a la mitad
    layers.Dropout(0.25),          # Regularización: apaga 25% de neuronas aleatoriamente

    # ── Bloque 2: características de nivel medio (formas, patrones)
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # ── Bloque 3: clasificación (capas densas)
    layers.Flatten(),              # Convierte mapa de características a vector 1D
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(N_CLASES, activation='softmax')  # Salida: probabilidad por clase
], name='CNN_Tema3')

# ── Compilación ───────────────────────────────────────────────────────────────
modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ── Resumen de la arquitectura ────────────────────────────────────────────────
modelo.summary()
print(f'\nHiperparámetros:')
print(f'  Épocas          : {EPOCAS}')
print(f'  Batch size      : {BATCH_SIZE}')
print(f'  Learning rate   : {LEARNING_RATE}')
print(f'  Clases          : {N_CLASES}')

In [ ]:
# ─── (OPCIONAL) OPCIÓN B — Transfer learning con MobileNetV2 ─────────────────
# Descomenta y ejecuta ESTA celda en lugar de la anterior si quieres usar
# un modelo preentrenado. Útil cuando el dataset propio es pequeño.

# base = keras.applications.MobileNetV2(
#     input_shape=(96, 96, 3),   # MobileNetV2 requiere mínimo 96×96
#     include_top=False,          # Excluir las capas de clasificación originales
#     weights='imagenet'          # Pesos preentrenados en ImageNet (1.4M imágenes)
# )
# base.trainable = False          # Congelar: no reentrenar esas capas
#
# modelo = keras.Sequential([
#     layers.Resizing(96, 96),            # Ajusta al tamaño requerido por la base
#     base,
#     layers.GlobalAveragePooling2D(),
#     layers.Dense(128, activation='relu'),
#     layers.Dropout(0.3),
#     layers.Dense(N_CLASES, activation='softmax')
# ], name='MobileNetV2_TransferLearning')
#
# modelo.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=0.0005),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )
# modelo.summary()
print('Opción B disponible — descomenta el bloque superior para usarla.')

In [ ]:
# ── Entrenamiento del modelo ───────────────────────────────────────────────────
# Se registra el historial para graficar loss y accuracy por época
print('Iniciando entrenamiento...')
print('-' * 55)

historial = modelo.fit(
    x_train, y_train,
    epochs=EPOCAS,
    batch_size=BATCH_SIZE,
    validation_split=0.15,   # 15% de train como validación
    verbose=1
)

print('-' * 55)
print('Entrenamiento completado.')

In [ ]:
# ── Gráficas de entrenamiento: Loss y Accuracy por época ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Paso 4 — Historial de entrenamiento', fontsize=13, fontweight='bold')

# Pérdida
axes[0].plot(historial.history['loss'],     label='Entrenamiento', color='steelblue',  lw=2)
axes[0].plot(historial.history['val_loss'], label='Validación',    color='darkorange', lw=2, ls='--')
axes[0].set_title('Función de pérdida (Loss)')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Exactitud
axes[1].plot(historial.history['accuracy'],     label='Entrenamiento', color='steelblue',  lw=2)
axes[1].plot(historial.history['val_accuracy'], label='Validación',    color='darkorange', lw=2, ls='--')
axes[1].set_title('Exactitud (Accuracy)')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('paso4_historial.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: paso4_historial.png')

In [ ]:
# ── Evaluación del modelo en el conjunto de prueba ────────────────────────────
loss_test, acc_test = modelo.evaluate(x_test, y_test, verbose=0)

print('RESULTADOS EN CONJUNTO DE PRUEBA')
print('=' * 40)
print(f'  Loss (pérdida)  : {loss_test:.4f}')
print(f'  Accuracy        : {acc_test:.4f}  ({acc_test*100:.1f}%)')
print('=' * 40)

# Predicciones para la matriz de confusión
y_pred_prob = modelo.predict(x_test, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

# Reporte de clasificación completo
print('\nREPORTE DE CLASIFICACIÓN:')
print(classification_report(y_test, y_pred, target_names=NOMBRES_CLASES))

In [ ]:
# ── Matriz de confusión ───────────────────────────────────────────────────────
# Muestra verdaderos positivos, falsos positivos y falsos negativos por clase
mc = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=mc, display_labels=NOMBRES_CLASES)
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Paso 4 — Matriz de confusión\n(conjunto de prueba)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('paso4_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: paso4_confusion.png')

# Resumen interpretativo
print('\nINTERPRETACIÓN:')
for i, nombre in enumerate(NOMBRES_CLASES):
    tp = mc[i, i]
    total = mc[i].sum()
    print(f'  Clase "{nombre}": {tp}/{total} clasificadas correctamente ({tp/total*100:.1f}%)')

In [ ]:
# ── Visualización de predicciones individuales ────────────────────────────────
# Muestra ejemplos correctos e incorrectos para análisis visual
np.random.seed(0)
idx_muestra = np.random.choice(len(x_test), 16, replace=False)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Paso 4 — Predicciones del modelo (verde=correcto, rojo=error)',
             fontsize=12, fontweight='bold')

for ax, i in zip(axes.ravel(), idx_muestra):
    ax.imshow(x_test[i])
    real  = NOMBRES_CLASES[y_test[i]]
    pred  = NOMBRES_CLASES[y_pred[i]]
    conf  = y_pred_prob[i][y_pred[i]] * 100
    color = 'green' if y_test[i] == y_pred[i] else 'red'
    ax.set_title(f'Real: {real}\nPred: {pred} ({conf:.0f}%)', fontsize=7, color=color)
    ax.axis('off')

plt.tight_layout()
plt.savefig('paso4_predicciones.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: paso4_predicciones.png')

### 📝 Registro del Paso 4 _(completa esta celda)_

**Opción elegida:** ☐ A (CNN desde cero) &nbsp;&nbsp; ☐ B (Transfer learning)  
**Arquitectura utilizada:** ___  
**Épocas entrenadas:** ___ &nbsp;&nbsp; **Batch size:** ___ &nbsp;&nbsp; **Learning rate:** ___  

**Resultados:**

| Métrica | Valor |
|---|---|
| Accuracy en prueba | ___ % |
| Loss en prueba | ___ |
| Clase con mejor rendimiento | ___ |
| Clase con más errores | ___ |

**¿Qué indica la gráfica de loss? ¿Hay overfitting (sobreajuste)?** ___  
**¿Qué confusiones observas en la matriz? ¿Por qué crees que ocurren?** ___

---
## PASO 5 — Integración de un esquema básico de rastreo (tracking)
**Tiempo estimado: 2 horas**

El rastreo consiste en **seguir la identidad** de uno o varios objetos a lo largo del tiempo en un video.  
A diferencia de la detección (qué y dónde en un solo cuadro), el rastreo **mantiene el ID** entre cuadros sucesivos.

Flujo de esta sección:
```
Video / secuencia  →  Detección por cuadro  →  Asignación de ID  →  Visualización
(entrada)             (clasificador CNN)        (distancia euclidiana)  (cajas + etiquetas)
```

> 💡 Se implementa un tracker simple por **distancia euclidiana de centroides** — suficiente para objetos que no se ocluyen.

In [ ]:
# ── Clase Tracker: asigna IDs a objetos entre cuadros ────────────────────────
# Criterio de asociación: menor distancia euclidiana entre centros de cajas

class TrackerSimple:
    """
    Tracker por cercanía de centroides.
    - Mantiene un diccionario {id: centroide_anterior}.
    - En cada cuadro, asigna el ID al centroide más cercano.
    - Si ningún objeto previo está cerca (umbral), crea un nuevo ID.
    """
    def __init__(self, umbral_dist=60, max_desaparecidos=5):
        self.siguiente_id     = 0
        self.objetos          = {}       # {id: centroide}
        self.desaparecidos    = {}       # {id: cuadros sin detectar}
        self.umbral_dist      = umbral_dist
        self.max_desaparecidos = max_desaparecidos

    def actualizar(self, centroides_nuevos):
        """
        Recibe lista de centroides [(cx, cy), ...] del cuadro actual.
        Devuelve {id: centroide} actualizado.
        """
        # Sin detecciones: aumentar contador de desaparecidos
        if len(centroides_nuevos) == 0:
            for oid in list(self.desaparecidos):
                self.desaparecidos[oid] += 1
                if self.desaparecidos[oid] > self.max_desaparecidos:
                    del self.objetos[oid]
                    del self.desaparecidos[oid]
            return self.objetos

        # Sin objetos registrados: registrar todos como nuevos
        if len(self.objetos) == 0:
            for c in centroides_nuevos:
                self.objetos[self.siguiente_id] = c
                self.desaparecidos[self.siguiente_id] = 0
                self.siguiente_id += 1
            return self.objetos

        # Calcular distancias entre objetos existentes y nuevos centroides
        ids_existentes = list(self.objetos.keys())
        cents_existentes = np.array(list(self.objetos.values()), dtype='float')
        cents_nuevos     = np.array(centroides_nuevos,           dtype='float')

        # Matriz de distancias: fila=existente, col=nuevo
        dists = np.linalg.norm(
            cents_existentes[:, None] - cents_nuevos[None, :], axis=2
        )

        # Asignación greedy: el par con menor distancia primero
        filas = dists.min(axis=1).argsort()
        cols  = dists.argmin(axis=1)[filas]

        usados_f, usados_c = set(), set()
        for f, c in zip(filas, cols):
            if f in usados_f or c in usados_c:
                continue
            if dists[f, c] > self.umbral_dist:
                continue
            oid = ids_existentes[f]
            self.objetos[oid]       = centroides_nuevos[c]
            self.desaparecidos[oid] = 0
            usados_f.add(f); usados_c.add(c)

        # Objetos no asociados: incrementar desaparecidos
        for f, oid in enumerate(ids_existentes):
            if f not in usados_f:
                self.desaparecidos[oid] += 1
                if self.desaparecidos[oid] > self.max_desaparecidos:
                    del self.objetos[oid]
                    del self.desaparecidos[oid]

        # Nuevos centroides sin asignar: crear nuevos IDs
        for c_idx in range(len(centroides_nuevos)):
            if c_idx not in usados_c:
                self.objetos[self.siguiente_id] = centroides_nuevos[c_idx]
                self.desaparecidos[self.siguiente_id] = 0
                self.siguiente_id += 1

        return self.objetos

print('✅ Clase TrackerSimple definida correctamente.')
print('  Criterio de asociación : distancia euclidiana mínima entre centroides')
print(f'  Umbral de distancia    : 60 px (modifica en el constructor si necesitas ajustar)')

In [ ]:
# ── Generación de una secuencia sintética de cuadros para demostrar el rastreo ─
# Si tienes un video propio, reemplaza esta sección por cv2.VideoCapture()

def generar_secuencia(n_cuadros=20, alto=300, ancho=400, n_objetos=3, seed=7):
    """
    Crea una secuencia de imágenes con objetos de colores moviéndose.
    Devuelve lista de (frame_bgr, lista_de_cajas [(x, y, w, h, etiqueta), ...])
    """
    np.random.seed(seed)
    colores   = [(220, 80,  80),  (80, 180, 80),  (80, 100, 220)]  # BGR
    etiquetas = NOMBRES_CLASES[:n_objetos]
    radios    = np.random.randint(20, 35, n_objetos)
    posiciones = np.array([[np.random.randint(50, ancho-50),
                             np.random.randint(50, alto-50)] for _ in range(n_objetos)], dtype=float)
    velocidades = (np.random.rand(n_objetos, 2) - 0.5) * 10

    secuencia = []
    for _ in range(n_cuadros):
        frame = np.ones((alto, ancho, 3), dtype=np.uint8) * 230  # Fondo gris claro
        cajas = []
        for j in range(n_objetos):
            posiciones[j] += velocidades[j]
            # Rebotar en los bordes
            for d, lim in enumerate([ancho, alto]):
                if posiciones[j, d] < radios[j] or posiciones[j, d] > lim - radios[j]:
                    velocidades[j, d] *= -1
            cx, cy = int(posiciones[j, 0]), int(posiciones[j, 1])
            r = radios[j]
            cv2.circle(frame, (cx, cy), r, colores[j], -1)
            cajas.append((cx - r, cy - r, 2*r, 2*r, etiquetas[j]))
        secuencia.append((frame, cajas))
    return secuencia

N_OBJETOS_RASTREO = min(3, len(NOMBRES_CLASES))
secuencia = generar_secuencia(n_cuadros=20, n_objetos=N_OBJETOS_RASTREO)
print(f'✅ Secuencia generada: {len(secuencia)} cuadros — {N_OBJETOS_RASTREO} objeto(s) por cuadro')
print('  (Para usar tu propio video: cap = cv2.VideoCapture("tu_video.mp4"))')

In [ ]:
# ── Aplicar el tracker a la secuencia y visualizar cuadros representativos ────
tracker = TrackerSimple(umbral_dist=80, max_desaparecidos=5)
cuadros_procesados = []

COLORES_ID = [
    (255, 100, 100), (100, 255, 100), (100, 100, 255),
    (255, 200, 0),   (200, 0, 255),   (0, 200, 255)
]

for n_frame, (frame, cajas) in enumerate(secuencia):
    centroides = [(x + w//2, y + h//2) for (x, y, w, h, _) in cajas]
    objetos_activos = tracker.actualizar(centroides)

    frame_vis = frame.copy()

    # Dibujar cajas delimitadoras y etiquetas
    for (x, y, w, h, etiqueta) in cajas:
        cv2.rectangle(frame_vis, (x, y), (x+w, y+h), (50, 50, 50), 2)
        cv2.putText(frame_vis, etiqueta, (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (30, 30, 30), 1)

    # Dibujar IDs del tracker
    for oid, (cx, cy) in objetos_activos.items():
        cx, cy = int(cx), int(cy)
        color = COLORES_ID[oid % len(COLORES_ID)]
        cv2.circle(frame_vis, (cx, cy), 6, color, -1)
        cv2.putText(frame_vis, f'ID:{oid}', (cx + 8, cy - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

    # Número de cuadro
    cv2.putText(frame_vis, f'Cuadro {n_frame+1:02d}', (8, 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (40, 40, 40), 2)

    cuadros_procesados.append(cv2.cvtColor(frame_vis, cv2.COLOR_BGR2RGB))

# Mostrar cuadros seleccionados de la secuencia
indices_mostrar = [0, 4, 9, 14, 19]
fig, axes = plt.subplots(1, len(indices_mostrar), figsize=(16, 4))
fig.suptitle('Paso 5 — Rastreo de objetos (caja + etiqueta + ID)',
             fontsize=12, fontweight='bold')

for ax, idx in zip(axes, indices_mostrar):
    ax.imshow(cuadros_procesados[idx])
    ax.set_title(f'Cuadro {idx+1}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('paso5_tracking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: paso5_tracking.png')

In [ ]:
# ── (Opcional) Guardar la secuencia como GIF animado ─────────────────────────
try:
    import imageio
    gif_frames = [cv2.resize(f, (320, 240)) for f in cuadros_procesados]
    imageio.mimsave('paso5_tracking.gif', gif_frames, fps=6)
    print('✅ GIF guardado: paso5_tracking.gif')
except ImportError:
    print('imageio no disponible. Instala con: pip install imageio')
    print('El GIF es opcional; las capturas estáticas son suficientes para el reporte.')

### 📝 Registro del Paso 5 _(completa esta celda)_

**Entrada utilizada:** ☐ Secuencia sintética (código base) &nbsp;&nbsp; ☐ Video propio (`___`)  
**Número de cuadros procesados:** ___  
**Número de objetos rastreados:** ___  
**Umbral de distancia euclidiana utilizado:** ___ px  

**¿El ID de cada objeto se mantiene estable a lo largo de los cuadros?** ___  
**¿En qué cuadros observas cambios de ID no deseados (ID switching)?** ___  
**¿Qué criterio de asociación usa el TrackerSimple?** ___  
**¿Qué limitaciones tiene este tracker frente a oclusiones?** ___

---
## PASO 6 — Análisis de resultados y reflexión
**Tiempo estimado: 2 horas**

Redacta una reflexión estructurada sobre los resultados obtenidos en los Pasos 4 y 5,
y conecta lo aprendido con los Temas 1 y 2.

In [ ]:
# ── Resumen cuantitativo de resultados ────────────────────────────────────────
print('RESUMEN DE RESULTADOS — TEMA 3')
print('=' * 55)

print('\n[CLASIFICACIÓN CON RNC]')
print(f'  Accuracy en prueba  : {acc_test:.4f}  ({acc_test*100:.1f}%)')
print(f'  Loss en prueba      : {loss_test:.4f}')
print(f'  Épocas entrenadas   : {EPOCAS}')
print(f'  Clases              : {NOMBRES_CLASES}')

print('\n[DESEMPEÑO POR CLASE (matriz de confusión)]')
for i, nombre in enumerate(NOMBRES_CLASES):
    tp    = mc[i, i]
    total = mc[i].sum()
    print(f'  Clase "{nombre:<12}" : {tp}/{total} correctas ({tp/total*100:.1f}%)')

print('\n[RASTREO]')
print(f'  Cuadros procesados  : {len(cuadros_procesados)}')
print(f'  IDs asignados total : {tracker.siguiente_id}')
print(f'  Objetos esperados   : {N_OBJETOS_RASTREO}')
id_extra = tracker.siguiente_id - N_OBJETOS_RASTREO
print(f'  ID switching        : {max(0, id_extra)} (IDs extra = posibles reasignaciones)')
print('=' * 55)

### 📝 Reflexión final _(completa esta celda — mínimo ½ cuartilla)_

---

**1. Desempeño del modelo de clasificación:**  
_¿Qué clase(s) clasifica mejor el modelo y por qué crees que es así?_  
_¿En qué clase(s) se observa más confusión? ¿Qué características visuales hacen que esas clases sean difíciles de separar?_  
_¿Qué indican las métricas de accuracy, precisión y recall en tu problema?_  

---

**2. Comportamiento del rastreo:**  
_¿Qué tan estable fue el ID de cada objeto a lo largo de la secuencia?_  
_¿Qué problemas se presentaron (cambios de iluminación, velocidad alta, oclusiones)?_  
_¿Cómo mejorarías el tracker? (menciona al menos una alternativa: Kalman, SORT, DeepSORT...)_  

---

**3. Conexión con el Tema 1 (adquisición y representación):**  
_¿Cómo influyó el preprocesamiento (normalización, redimensionado) en el resultado del clasificador?_  
_¿Qué pasaría si las imágenes de entrada tuvieran mala iluminación o resolución insuficiente?_  

---

**4. Conexión con el Tema 2 (filtros y convolución):**  
_¿Qué relación ves entre los filtros Sobel/Gaussiano del Tema 2 y los filtros aprendidos por las capas convolucionales de la CNN?_  
_¿Por qué las primeras capas de una CNN aprenden filtros similares a los detectores de bordes que viste en el Tema 2?_  

---

**5. Aplicaciones reales:**  
_Menciona al menos 2 aplicaciones concretas donde esta combinación (clasificación + rastreo con RNC) sería útil en un entorno industrial, médico o educativo._

---
## REPORTE TÉCNICO — Portafolio de evidencias
_(Esta sección corresponde al Paso 7 del Manual del Participante)_

**Instrucciones:** Completa todos los campos. Este reporte, junto con las imágenes generadas, forma tu portafolio de evidencias del Tema 3.

---

### PORTADA

| | |
|---|---|
| **Nombre** | ___ |
| **Curso** | Redes Neuronales Convolucionales aplicadas a la Visión Computacional |
| **Tema** | 3 — Aplicaciones de redes neuronales convolucionales en visión |
| **Actividad** | T3.A3.1 — Implementación de clasificación y rastreo de objetos con RNC |
| **Fecha** | ___ |

---

### 1. Descripción del problema y contexto

| Campo | Valor |
|---|---|
| Contexto de aplicación | ___ |
| Objetos a clasificar | ___ |
| Clases definidas | Clase 0: ___ / Clase 1: ___ / Clase 2: ___ |
| Objetos rastreados | ___ |
| Propósito práctico | ___ |

---

### 2. Descripción del dataset

| Campo | Valor |
|---|---|
| Nombre / fuente | ___ |
| Número de clases | ___ |
| Imágenes de entrenamiento | ___ |
| Imágenes de prueba | ___ |
| Resolución | ___ × ___ px |
| Canales | ___ |
| Normalización aplicada | ___ |

---

### 3. Descripción del modelo de clasificación

| Parámetro | Valor |
|---|---|
| Opción elegida | ☐ A (desde cero) &nbsp; ☐ B (transfer learning) |
| Arquitectura | ___ |
| Número de parámetros | ___ |
| Épocas | ___ |
| Batch size | ___ |
| Learning rate | ___ |
| Accuracy final (prueba) | ___ % |
| Loss final (prueba) | ___ |

_Incluye las capturas: `paso4_historial.png`, `paso4_confusion.png`, `paso4_predicciones.png`_

---

### 4. Análisis comparativo — matriz de confusión

| Clase | Correctas | Total | Recall (%) | Principal confusión con |
|---|---|---|---|---|
| ___ | ___ | ___ | ___ | ___ |
| ___ | ___ | ___ | ___ | ___ |
| ___ | ___ | ___ | ___ | ___ |

---

### 5. Descripción del módulo de rastreo

| Campo | Valor |
|---|---|
| Método de rastreo | Distancia euclidiana de centroides (TrackerSimple) |
| Umbral de distancia | ___ px |
| Cuadros procesados | ___ |
| IDs totales asignados | ___ |
| Estabilidad observada | ___ |

_Incluye la captura: `paso5_tracking.png`_

---

### 6. Reflexión final _(mínimo ½ cuartilla — completa en la celda del Paso 6)_

---

**Formato de entrega:** `T3_A3_ApellidoNombre.pdf` — máximo 2 MB  
**Plataforma:** Aula Virtual TecNM → Tarea Tema 3  
**Complemento (si el facilitador lo solicita):** enlace al notebook de Colab

---
## EVALUACIÓN FORMATIVA — Cuestionario en plataforma
**Peso: 15% | Tiempo: 1 hora**

El cuestionario se responde en el **Aula Virtual TecNM**. Temas evaluados:

| # | Tema |
|---|---|
| 1-2 | Visión monocular: tareas (clasificación, detección, segmentación) |
| 3-4 | Visión estereoscópica: paralaje, disparidad, rectificación |
| 5-6 | Arquitecturas CNN: LeNet, ResNet, criterios de selección |
| 7-8 | Componentes de una CNN: capas convolucionales, pooling, softmax |
| 9  | Métricas: accuracy, matriz de confusión, precision, recall, F1 |
| 10 | Rastreo: diferencia con detección, ID switching, métricas de tracking |

> 10 reactivos de opción múltiple · **1 hora** · **2 intentos** · Se considera la calificación más alta

---
**Recursos de apoyo para el cuestionario:**
- Presentación "Tema 3. Aplicaciones de redes neuronales convolucionales en visión"
- Manual del Participante – Tema 3
- Este notebook con tus resultados y reflexiones

In [ ]:
# ── Resumen de archivos generados ─────────────────────────────────────────────
archivos = [
    ('paso3_dataset.png',       'Muestras del dataset por clase'),
    ('paso4_historial.png',     'Gráfica loss y accuracy por época'),
    ('paso4_confusion.png',     'Matriz de confusión'),
    ('paso4_predicciones.png',  'Predicciones individuales del modelo'),
    ('paso5_tracking.png',      'Cuadros del rastreo con IDs'),
    ('paso5_tracking.gif',      'Animación del tracking (opcional)'),
]
print('ARCHIVOS GENERADOS PARA EL PORTAFOLIO:')
print('-' * 60)
for archivo, descripcion in archivos:
    existe = '✅' if os.path.exists(archivo) else '⬜  (aún no generado)'
    print(f'  {existe}  {archivo:<30} {descripcion}')
print('-' * 60)
print('\nSiguiente paso: exporta este notebook como PDF')
print('Nombre de entrega: T3_A3_ApellidoNombre.pdf')